<a href="https://colab.research.google.com/github/lapshinaaa/recsys-tasks/blob/main/DeepRecSys2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep RecSys Course
## Notebook №2

In this notebook, we'll implement various loss functions that are commonly used for training Two-Tower models for CandGen.

### Data
Data are stored in `data.zip`, which consists of:
* `interactions.parquet` - user-item interactions from Yambda dataset (likes for the 500m version)
* `embeddings.parquet` - already filtered and more densely packed embeddings of tracks from Yambda
* `artists.parquet` - items' metadata with mapping into artistsм


In this task, we'll only be interested in `interactions.parquet`

The archive can be downloaded: [from here](https://drive.google.com/file/d/1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS/view?usp=sharing). We're downloading this in the next cell block so there's no need to use the link.

In [1]:
!pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -oq dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=0f211373-882a-4b81-9f19-ae0d2c56cfe1
To: /content/dataset.zip
100% 356M/356M [00:06<00:00, 51.9MB/s]


In [2]:
from collections import defaultdict
import copy
import gc
import os
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import polars as pl
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import tests
import math

# 0. Data Prep and Metrics

Data processing

In [3]:
# paths to data
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")

# global vars
TOPK = 100
CORE_MIN_INTERACTIONS_PER_ITEM = 5
TEST_INTERVAL_SECONDS = 7 * 24 * 60 * 60

# for reproducibility
np.random.seed(42)

data = pl.read_parquet(PATH_INTERACTIONS)
embeddings = pl.read_parquet(PATH_EMBEDDINGS)
artists = pl.read_parquet(PATH_ARTISTS)

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################


Metrics

In [4]:
def get_metrics(targets: List[int], candidates: List[int], topk: int) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################

    """
    Per-user metrics. targets = relevant items G_u, candidates = ranked list R_u (length >= topk).
    Returns hitrate@k(u), recall@k(u), ndcg@k(u).
    """

    recs = candidates[:topk]
    gt = set(targets)

    hits = [1 if item in gt else 0 for item in recs] # faster since gt is a dict
    num_hits = sum(hits)

    # Hitrate@K(u)
    hitrate = 1.0 if num_hits > 0 else 0.0

    # Recall@K(u)
    denom = min(len(targets), topk) # for len can't use len, MUST use original targets
    recall = (num_hits / denom) if denom > 0 else 0.0

    # DCG@K(u)
    dcg = 0.0
    for idx, h in enumerate(hits, start=1):  # idx = 1..K
        if h:
            dcg += 1.0 / math.log2(idx + 1)

    # iDCG@K(u)
    idcg = 0.0
    for idx in range(1, denom + 1):
        idcg += 1.0 / math.log2(idx + 1)

    ndcg = (dcg / idcg) if idcg > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg}


def evaluate(
    targets: Dict[int, List[int]],
    candidates: Dict[int, List[int]],
    catalog_size: int,
    topk: int = 100,
) -> Dict[str, float]:
    ###########################
    ### ╰( ͡° ͜ʖ ͡° )つ──☆*:・ﾟ
    ###########################
    """
    Aggregates metrics across users and computes coverage@K.
    Assumes candidates[uid] has length at least topk (or exactly topk, as your note says).
    """
    uids = list(targets.keys())

    hitrate_sum = 0.0
    recall_sum = 0.0
    ndcg_sum = 0.0

    # coverage: union of all recommended items across users in topK
    covered_items = set()

    for uid in uids:
        gt_u = targets[uid]
        rec_u = candidates[uid][:topk]  # guarantee topk slice

        m = get_metrics(gt_u, rec_u, topk=topk)
        hitrate_sum += m["hitrate"]
        recall_sum += m["recall"]
        ndcg_sum += m["ndcg"]

        covered_items.update(rec_u)

    num_users = len(uids)
    hitrate = hitrate_sum / num_users if num_users > 0 else 0.0
    recall = recall_sum / num_users if num_users > 0 else 0.0
    ndcg = ndcg_sum / num_users if num_users > 0 else 0.0

    coverage = (len(covered_items) / catalog_size) if catalog_size > 0 else 0.0

    return {"hitrate": hitrate, "recall": recall, "ndcg": ndcg, "coverage": coverage}

# 1. Dataset creation and collate func

In this task, you must implement some helper functions to work with user histories of variable length. These functions will later be used when building samplesm batches and when training models, so it is important to ensure a certain format of sequences.

#### Data format: flatten-representation of user history

Instead of storing the history of each user as a separate list (and then do padding to the unified len), we'll store the history of a given batch in one flattened tensor - `flatten` format:

`[u1_t1, u1_t2, ..., u1_tL1, u2_t1, ..., u2_tL2, ...]`

Essentially, in one tensor we'll write interactions of user 1, user 2, etc.

So that we don't lose the boundaries between the users, we'll separately store tensor `length`, where a number of elements of a given history will be stored (for each user in batch):

`length = [L1, L2, ..., LB]`

where `B` — batch size, а `Li` — history len of `i` user.


In this format, we'll store user history (sequence of interactions), but our task is to implement functions that will:
- restore boundaries of sequences using `length`,
- turn flatten-representation into a padded format + mask,
- prepare batch for feeding into a model.

### Function `create_masked_tensor`

Implement function `create_masked_tensor`, which will, using `flatten` representation of the batch of sequences and their lens, form `padded` tensor and bool mask of elements' positions.

First, we'll implement a func that will get lengths and return a tensor with boolean values (`(batch_size, max_len)`).

In [5]:
def get_mask(lengths: torch.Tensor) -> torch.Tensor:
  """
    Creates a boolean mask for variable-length sequences.
  """

 # batch_size = lengths.size(0)
  max_len = lengths.max().item()

  positions = torch.arange(max_len, device=lengths.device)
  mask = positions.unsqueeze(0) < lengths.unsqueeze(1)

  return mask

In [6]:
def create_masked_tensor(data: torch.Tensor, lengths: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
  """
  Converts a batch of flattened variable-length sequences into a padded tensor and mask.
  Supports:
    - indices: data shape (total_num_elements,)
    - embeddings/features: data shape (total_num_elements, d1, d2, ...)

  Parameters
  ----------
  data : torch.Tensor
      Input tensor containing flattened sequences:
      - For indices: shape (total_num_elements,)
      - For embeddings: shape (total_num_elements, embedding_dim)
  lengths : torch.Tensor
      1D tensor of sequence lengths, shape (batch_size,). Specifies the actual length
      of each sequence.

  Returns
  -------
  Tuple[torch.Tensor, torch.Tensor]
      - padded_tensor: Padded tensor of shape:
          - (batch_size, max_seq_len) for indices
          - (batch_size, max_seq_len, embedding_dim) for embeddings
          Shorter sequences are right-padded with zeros.
      - mask: Boolean mask of shape (batch_size, max_seq_len) where True indicates
          valid elements and False indicates padding. Can be used in attention or loss computation.

  Examples
  --------
  >>> data = torch.tensor([1, 2, 3, 4, 5, 6])  # sequences: [1,2], [3,4,5], [6]
  >>> lengths = torch.tensor([2, 3, 1])
  >>> padded, mask = create_masked_tensor(data, lengths)
  >>> padded
  tensor([[1, 2, 0],
          [3, 4, 5],
          [6, 0, 0]])
  >>> mask
  tensor([[ True,  True, False],
          [ True,  True,  True],
          [ True, False, False]])
  """
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################

  mask = get_mask(lengths)
  batch = lengths.size(0)
  elem_max = int(lengths.max().item())

  if data.dim() == 1:
    padded = torch.zeros(batch, elem_max, dtype=data.dtype, device=data.device)
  else:
    D = data.shape[1:]
    padded = torch.zeros(batch, elem_max, *D, dtype=data.dtype, device=data.device)

  start = 0
  for i, length in enumerate(lengths.tolist()):
    end = start + length
    padded[i, :length] = data[start:end]
    start = end

  return padded, mask

In [7]:
tests.test_create_masked_tensor(create_masked_tensor)

All good! :)


### Class `YambdaDataset`

Implement class `YambdaDataset`, which works with user interaction histories and prepares samples for further model training.

Dataset must support two modes of work, which is flagged by `is_train`, and also cutting the history up to the last `max_seq_len` elements.

In train-mode we turn one user history of length `T` into `T-1` training samples. That is, for a user with history `[i1, i2, ..., iT]` we're creating samples with prefixes:

- `history[:1] -> label = i2`
- `history[:2] -> label = i3`
- ...
- `history[:T-1] -> label = iT`

If we implement it brute force and in `__init__` materialize all such samples, we'll have a lot of data duplicates: the exact same item `i1` will be reappearing in almost all samples, `i2` — in all of them except the first one, etc.

That is why in `__init__` we're storing only indexes/pointers, and the prefix itself of `history` and truncation we're building as we go in `__getitem__`.

#### Inputs

- `histories: Dict[uid, List[int]]` — time-ordered histories of user interactions.
- `labels: Dict[uid, List[int]]` — target items for a user for evaluation (last week in our case).
- `is_train: bool` — dataset mode of work.
- `max_seq_len: int` — max length of a returned history (set to default of `100`).

#### Mode 1: Train mode (`is_train=True`)

In train mode the dataset must prepare samples for user in terms of next-item prediction.

If a user history is: `[i1, i2, ..., iT]`, then `T - 1` samples must be created. For each `t` from `1` to `T-1` (next item position):

- `history` = prefix `history[:t]`, truncated to the last `max_seq_len` elements
- `label` = next item `history[t]`

Format of a train-sample:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  },
  "label": int
}
```

#### Mode 2: Inference mode (`is_train=False`)

In evaluation mode our dataset must return exactly one sample per user.
User is in a dataset only if there are targets for them in `labels`.
Sample contents:
- `history` = user history, truncated to the last `max_seq_len` elements.

Format of an eval-sample:
```python
{
  "uid": uid,
  "history": {
    "item_id": List[int],
    "length": int
  }
}
```

In [8]:
class YambdaDataset(Dataset):
  """
  PyTorch Dataset for user interaction histories with next-item prediction samples.

  Parameters
  ----------
  histories : Dict[Any, List[int]]
      Mapping from user id to a list of interacted item ids (sorted by time).
  labels : Dict[Any, List[int]]
      Mapping from user id to a list of target item ids.
      Used only to filter users in eval mode (`uid in labels`).
  is_train : bool
      If True, generate multiple (prefix, next_item) samples per user.
      If False, return one sample per user (filtered by presence in `labels`).
  max_seq_len : int, default 100
      Maximum number of most recent items to keep in the returned history.

  Returns
  -------
  Dict[str, Any]
      Train mode (`is_train=True`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
            "label": int,
          }

      Eval mode (`is_train=False`):
          {
            "uid": uid,
            "history": {"item_id": List[int], "length": int},
          }

      where:
        - history["item_id"] contains up to `max_seq_len` last items of the selected prefix/history
        - history["length"] is the length of the returned (possibly truncated) history
        - label is a single next item id (int)

  Examples
  --------
  Train mode:
  >>> ds = YambdaDataset(histories, labels={}, is_train=True, max_seq_len=100)
  >>> s = ds[0]
  >>> s["uid"]
  >>> s["history"]["item_id"], s["history"]["length"]
  >>> s["label"]

  Eval mode (filters users by `labels` keys):
  >>> ds = YambdaDataset(histories, labels=test_targets, is_train=False)
  >>> s = ds[0]
  >>> s["uid"]
  >>> s["history"]["item_id"], s["history"]["length"]
  """

  def __init__(
      self,
      histories: Dict[Any, List[int]],
      labels: Dict[Any, List[int]],
      is_train: bool,
      max_seq_len: int = 100,
  ) -> None:
      super().__init__()
      self.histories = histories
      self.labels = labels
      self.is_train = is_train
      self.max_seq_len = max_seq_len
      #####################
      ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
      #####################

      self.samples = []

      if self.is_train:
          for uid, hist in self.histories.items():
              for t in range(1, len(hist)):
                  self.samples.append((uid, t))
      else:
          for uid in self.histories: # we're iterating over keys
              if uid in self.labels:
                  self.samples.append(uid)


  def __len__(self) -> int:
      """Return number of samples (prefix samples in train mode, users in eval mode)."""
      #####################
      ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
      #####################
      return len(self.samples)

  def __getitem__(self, idx: int) -> Dict[str, Any]:
      """
      Build and return a single sample using an index pointer (uid, t).

      In train mode: returns a truncated prefix and the next item as an integer label.
      In eval mode: returns the truncated full history.
      """
      #####################
      ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
      #####################
      if self.is_train:
        uid, t = self.samples[idx]
        full_history = self.histories[uid]

        history = full_history[:t]
        history = history[-self.max_seq_len:]
        label = full_history[t]

        return {
            "uid": uid,
            "history": {
                "item_id": history,
                "length": len(history),
            },
            "label": label,
        }

      else:
        uid = self.samples[idx]
        full_history = self.histories[uid]
        history = full_history[-self.max_seq_len:]

        return {
            "uid": uid,
            "history": {
                "item_id": history,
                "length": len(history),
            },
        }

In [9]:
tests.test_yambda_dataset(YambdaDataset)

All good! :)


### Function `collate_fn`

Implement the function `collate_fn`, which will be used in `DataLoader` for forming lists of samples
from `YambdaDataset` into batches, that are convenient for passing into the model and working with.

As stated earlier, we're using flatten-representation: instead of padding to the same length we
1) concatinate all user histories from a batch into one 1D-tensor  
2) separately save `length`, so that later we can restore boundaries of sequences


#### Input

`batch: List[Dict[str, Any]]` — list of samples from `YambdaDataset`.

#### What `collate_fn` must do

The function must form a dictionary, where all the elements are `torch.Tensor` of type `torch.long`.

- `result["history"]["item_id"]` 1D tensor, got by concatination all `history["item_id"]` in order of objects in `batch` dim: `(sum(history_lengths),)`

- `result["history"]["length"]` 1D tensor of lengths of histories for each object in batch of dim : `(batch_size,)`

- `result["uid"]` 1D tensor of identifiers of users of dim: `(batch_size,)`

- `result["label"]` (only if input samples contain `"label"`) 1D tensor of lables (next item id) in order of objects in batch dim: `(batch_size,)`

#### Requirements

- Don't use `padding`. Only `flatten`-concatination + `lengths`.
- Store the order of objects in `batch` when concatinating.
- Return `"label"` only it exists in input samples.
- All numeric values must be converted into `torch.Tensor` of type `torch.long`.


In [10]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
  """
  Collate function that converts a list of samples into a **flatten** batch representation.

  This function implements the "flatten" batching scheme: instead of padding variable-length
  sequences to a common length, it concatenates all user histories in the batch into a single
  1D tensor and returns a companion `length` tensor to recover per-user boundaries later.

  The function is compatible with `YambdaDataset` in two modes:
    - Train mode samples contain keys: `"uid"`, `"history"`, and `"label"` (where `"label"` is an `int`).
    - Eval mode samples contain keys: `"uid"` and `"history"`.

  Output batch format
  -------------------
  The returned dictionary contains:
    - `result["history"]["item_id"]`: 1D tensor with all history items concatenated in the
      order of samples in `batch`, shape `(sum(history_lengths),)`, dtype `torch.long`.
    - `result["history"]["length"]`: 1D tensor of per-sample history lengths,
      shape `(batch_size,)`, dtype `torch.long`.
    - `result["uid"]`: 1D tensor of user ids, shape `(batch_size,)`, dtype `torch.long`.
    - If `"label"` is present in the input samples (train batches):
        - `result["label"]`: 1D tensor of labels (next item ids), shape `(batch_size,)`,
          dtype `torch.long`.

  Parameters
  ----------
  batch : List[Dict[str, Any]]
      List of samples returned by the dataset `__getitem__`.

  Returns
  -------
  Dict[str, Any]
      A nested dictionary where all returned values are `torch.Tensor` objects.

  Examples
  --------
  - Train-mode: returns `"history"` + `"uid"` + `"label"` (1D tensor of next-item ids).
  - Eval-mode: returns `"history"` + `"uid"` (no `"label"` key).
  """
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################

  uids=[]
  all_items=[]
  lengths=[]
  labels=[]

  has_label = "label" in batch[0] # flag for train or eval

  for sample in batch: # iterating over each sample (from __getitem__) from our batch (each one is a dict)
    uids.append(sample["uid"])
    all_items.extend(sample["history"]["item_id"])
    lengths.append(sample["history"]["length"])

    if has_label:
      labels.append(sample["label"])

  result = {
      "uid": torch.tensor(uids, dtype=torch.long),
      "history": {
          "item_id": torch.tensor(all_items, dtype=torch.long),
          "length": torch.tensor(lengths, dtype=torch.long),
      }
  }

  if has_label:
    result["label"] = torch.tensor(labels, dtype=torch.long)

  return result

In [11]:
tests.test_collate_fn(collate_fn)

All good! :)


# 2. Implementation of the computation graph for training a two-tower model (without the loss) and for inference (candidate retrieval).

## UserEncoder

Implement class `UserEncoder`, which is the main component of our model.

`UserEncoder` — is a module, which based on user's interaction history builds their context representation.

Each user $u$ is described by their history of interactions $S_u$.

For each item $i$ from catalog there is a trainable embedding $e_i \in \mathbb{R}^d$.

Representation for a user $u$, $P_u$, is an aggregate of embeddings of all their prior interactions. In this task, as an aggregation bag-of-words-like representation of a user must be implemented based on their history of interactions.

For user $u$ with the history of interactions $i_1, i_2, \ldots, i_{|S_u|}$ , the following must be received:

$$
P_u = \sum_{k=1}^{|S_u|} e_{i_k}.
$$

#### Model input

During training data from `YambdaDataset` using `collate_fn` is transformed into `flatten`-batches and `batch["history"]` is passed as input to the method `UserEncoder.forward` to get user representations from the batch.

#### What the model must do

1. Transform received `item_id` into object embeddings;
2. Calculate representations of user as a tensor of size `(batch_size, embedding_dim)`.
3. Return these representations

In [16]:
class UserEncoder(nn.Module):
  """
  User encoder that represents each user by a cumulative prefix sum of item embeddings.

  Parameters
  ----------
  num_items : int
      Total number of unique items in the catalog.
      Item ids must be in ``[0, num_items - 1]``.
  embedding_dim : int
      Dimension of item embeddings.

  Forward input
  -------------
  inputs : Dict[str, torch.Tensor]
      Dictionary with keys:
      - "item_id": Flattened item indices for concatenated sequences,
        shape ``(total_num_events,)``, dtype ``torch.long``.
      - "length": Per-user sequence lengths, shape ``(batch_size,)``,
        dtype ``torch.long``.

  Forward output
  --------------
  torch.Tensor
      User representations, one vector per user, shape ``(batch_size, embedding_dim)``.
  """
  def __init__(self, num_items: int, embedding_dim: int) -> None:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    super().__init__()
    self.item_embeddings = nn.Embedding(num_items, embedding_dim)

  def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    item_id = inputs["item_id"]
    lengths = inputs["length"]
    embeddings = self.item_embeddings(item_id)

    user_representations = []
    start = 0

    for length in lengths.tolist():
      end = start + length
      user_representations.append(embeddings[start:end].sum(dim=0)) # resulting embedding of a user gotten by sum
      start = end

    return torch.stack(user_representations) # just stack all the resulting embeddings of users

In [17]:
tests.test_user_encoder(UserEncoder)

All good! :)


## TwoTowerModel: training and inference

`TwoTowerModel` объединяет `UserEncoder` и логику обучения/инференса модели.

#### Обозначения

$\mathbf{E} \in \mathbb{R}^{|I| \times d}$ — таблица эмбеддингов айтемов

$\mathbf{P}_u \in \mathbb{R}^d$ — представление пользователя $u$

Релевантность айтема $i$ для пользователя $u$: $r_i = \langle \mathbf{E}_{i}, \mathbf{P}_{u}\rangle$.


#### Что должна делать модель

Нам дан батч:
`inputs["history"]`: история (то, на основе чего строим пользователя и обучаетмся)
`inputs["labels"]`: таргеты/позитивы (айтемы с последней недели, по которым хотим получать метрики на эвале)

Для каждого пользователя в батче:
- строим $\mathbf{U}$ по его истории

#### Режим обучения (`self.training == True`)

- прогнать `inputs["history"]` через `UserEncoder` и получить $\mathbf{U}$ для пользователей в батче
- вычислить лосс через метод `compute_loss`
- вернуть лосс

#### Режим эвала (`self.training == False`):

- прогнать `inputs["history"]` через `UserEncoder` и получить $\mathbf{U}$ для пользователей в батче
- посчитать: $\text{all\_scores} = \langle\mathbf{U}, \mathbf{E}^{\top}\rangle$ размера `(batch_size, num_items)`
- вернуть тензор `all_scores` (метрики считаются отдельно)


#### Откуда берутся позитивы на обучении

Обучение формулируется как задача `next item prediction`. Для каждого шага в пользовательской истории позитивным примером считается следующий айтем в последовательности пользователя. Иными словами, модель обучается предсказывать следующий объект взаимодействия на основе всех предыдущих.
    
    

In [ ]:
class TwoTower(nn.Module):
  """
  Recommendation model combining user encoder with training and inference logic.

  The model produces:
    - a user representation vector `P_u` via `UserEncoder`
    - an item representation matrix `E` from the embedding table
    - uses dot-product relevance scores: `r_{ui} = <P_u, E_i>`.

  The `forward` method behaves differently depending on `self.training`:

  Training mode (`self.training == True`)
    - Encodes users.
    - Delegates loss computation to `compute_loss(...)`.
    - Returns a loss tensor.

  Evaluation / inference mode (`self.training == False`)
    - Encodes users.
    - Computes scores against all items in the catalog.
    - Returns a full score matrix.

  Parameters
  ----------
  num_items : int
    Total number of unique items in the catalog. Item ids must be in `[0, num_items - 1]`.
  embedding_dim : int
    Dimension of user/item embeddings.

  Notes
  -----
  This base class does not implement `compute_loss`. Subclasses should override it to define a training objective.
  """

  def __init__(self, num_items: int, embedding_dim: int) -> None:
    super().__init__()
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.init_weights(0.02)

  @torch.no_grad()
  def init_weights(self, initializer_range: float) -> None:
    """
    Initialize all model parameters with truncated normal distribution.

    Parameters
    ----------
    initializer_range : float
        Standard deviation of the truncated normal initializer.
    """
    for key, value in self.named_parameters():
      assert "weight" in key
      nn.init.trunc_normal_(
        value.data,
        std=initializer_range,
        a=-2 * initializer_range,
        b=2 * initializer_range,
      )

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    """
    Compute training loss.

    Parameters
    ----------
    user_repr : torch.Tensor
        User representations returned by the encoder, shape ``(batch_size, embedding_dim)``.
    inputs : Dict[str, Any]
        Full input batch. Expected to contain at least:
          - ``inputs["history"]``: dict with flattened history fields
          - label information (e.g., ``inputs["label"]``), depending on the training setup

    Returns
    -------
    torch.Tensor
        Scalar loss tensor.
    """
    # Эту функцию мы реализуем отдельно позже!
    # Не трогать ее и не менять здесь!
    raise NotImplementedError

  def forward(self, inputs: Dict[str, Any]) -> Dict[str, torch.Tensor]:
    """
    Run a forward pass with mode-dependent behavior.
    During training: computes and returns loss.
    During evaluation: computes and returns ranking scores for all items.

    Parameters
    ----------
    inputs : Dict[str, Any]
        Batch dictionary produced by `collate_fn`. Expected keys:
          - ``"history"``: dict with
                - ``"item_id"``: 1D flattened history item ids
                - ``"length"``: per-user history lengths
          - ``"uid"``: user ids tensor

    Returns
    -------
    torch.Tensor
        - If training (self.training == True): loss, scalar tensor
        - If evaluating (self.training == False): all_scores, relevance scores for all items with shape (batch_size, num_items)
    """
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
tests.test_two_tower(TwoTower)

Посмотрим на то, что у нас получилось

In [ ]:
TRAIN_BATCH_SIZE = 2048
EVAL_BATCH_SIZE = 2048


catalog_size = data["item_id"].n_unique()

train_histories = dict(
    train_df.group_by("uid")
    .agg(pl.col("item_id").sort_by("timestamp"))
    .iter_rows()
)

test_targets = dict(
    test_df.group_by("uid")
    .agg(pl.col("item_id"))
    .iter_rows()
)


yambda_train_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=True
)

yambda_eval_dataset = YambdaDataset(
  histories=train_histories,
  labels=test_targets,
  is_train=False
)

yambda_train_dataloader = DataLoader(
  dataset=yambda_train_dataset,
  batch_size=TRAIN_BATCH_SIZE,
  shuffle=True,
  collate_fn=collate_fn,
  drop_last=True
)

yambda_eval_dataloader = DataLoader(
  dataset=yambda_eval_dataset,
  batch_size=EVAL_BATCH_SIZE,
  shuffle=False,
  collate_fn=collate_fn,
  drop_last=False
)

# 3. Цикл обучения (1 балл)

Реализуйте функцию `evaluation`, которая выполняет оценку качества модели рекомендаций.

Функция должна:
1) Получить top-k рекомендаций для каждого пользователя из `dataloader`.
2) Собрать их в словарь формата `Dict[uid, List[item_id]]`.
3) Посчитать метрики, вызвав `evaluate(...)`, и вернуть результат.

#### Tips & Tricks
- Не забудьне перевести модель в `.eval()` режим
- Метрики можно считать с помощью функции `evaluate` из ДЗ 1

In [ ]:
def evaluation(
  dataloader: DataLoader,
  model: TwoTower,
  catalog_size: int,
  topk: int,
  device: str = "cuda",
) -> Dict[str, float]:
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################
  pass


Реализуйте функцию `train`, которая обучает модель и после каждой эпохи запускает валидацию.

После завершения каждый эпохи необходимо:
- Запустить функцию `evaluation` на `valid_dataloader`
- Вывести метрики валидации в читаемом виде.
- Посчитать и вывести средний лосс за эпоху.

После окончания обучения вывести сообщение о завершении и вернуть состояние модели (`state dict`).

#### Примечания

- Важно корректно переключать режимы модели:
  - обучение выполняется в `train` режиме,
  - валидация должна выполняться внутри `evaluation`, где модель переводится в `eval` режим.
- Перенос батча на `device` должен корректно работать со структурой батча, где могут встречаться вложенные словари с тензорами.

In [ ]:
def train(
    train_dataloader: DataLoader,
    valid_dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    catalog_size: int,
    topk: int,
    device: str = "cuda"
  ) -> Dict[str, Any]:
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################
  pass

# Реализуем различные способы обучения полученной двухбашенной модели

In [ ]:
NUM_EPOCHS = 1
LEARNING_RATE = 1e-3
DEVICE = "cuda"

## 4. Softmax loss (1 балл)



В задаче отбора кандидатов каждому пользователю и каждому айтему сопоставляется векторное представление размерности $d$ в общем латентном пространстве.

Скор релевантности пользователя $u$ и айтема $i$ вычисляется как скалярное произведение их эмбеддингов:

$$
r(u, i) = \langle \mathbf{E}_i,\;\mathbf{P}_u \rangle,
$$

где:
- $\mathbf{P}_u \in \mathbb{R}^d$ — представление пользователя, полученное из `UserEncoder`;
- $\mathbf{E}_i \in \mathbb{R}^d$ — обучаемое представление айтема.

Чем больше значение $r(u, i)$, тем более релевантным считается айтем $i$ для пользователя $u$.

На этапе инференса айтемы ранжируются по убыванию релевантности, и модель возвращает top-K кандидатов.

В этом задании мы обучаем модель на задачу экстремальной многоклассовой классификации.

Для каждого пользователя в батче нужно предсказать один правильный айтем из всего каталога айтемов размера $|\mathcal{I}|$.

#### Формула

Пусть $i^+$ — следующий айтем для пользователя $u$, тогда:

$$
\mathcal{L}_{\text{softmax}} = - \sum_{u \in \mathbf{U}} \log p(i^+ \mid u) = - \sum_{u \in \mathbf{U}} \left[r(u, i^+) - \log \sum_{j\in\mathcal{I}} \exp(r(u,j))\right].
$$


#### Почему это лучший способ обучать модели для этой стадии

Full softmax использует информацию обо всём каталоге: обучение модели эквивалентно применению: мы ищем позитив из всего каталога на обучении, мы берем top-K айтемов из всего каталога на применении.

In [ ]:
class SoftmaxModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
tests.test_softmax_model(SoftmaxModel)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_full = SoftmaxModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_full = torch.optim.Adam(params=model_full.parameters(), lr=LEARNING_RATE)
best_checkpoint_full  = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_full,
    optimizer=optimizer_full,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_full.load_state_dict(best_checkpoint_full)
final_metrics_full = evaluation(
    yambda_eval_dataloader,
    model_full,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_recs(final_metrics_full)

## 5. BCE loss (1 балл)

Главная проблема предыдущего подхода — вычисления и память: полный softmax обычно применим, когда каталог не слишком большой — примерно до десятков/сотен тысяч айтемов. Для каталогов в миллионы обычно используют более простые подходы. Пойдет по их усложнению. Самый простой из них Binary Cross-Entropy (BCE).

Для каждого пользователя $u$ мы рассматриваем:

- позитивный пример: айтем $i^+$, с которым пользователь действительно взаимодействовал (следующий после истории пользователя);
- негативные примеры: айтемы $i^-$, сэмплированные из каталога (обычно равномерно), с которыми пользователь не взаимодействовал.

Модель обучается предсказывать вероятность того, что айтем является релевантным для пользователя в данный момент времени.

#### Формула

Для одного пользователя $u$, позитивного айтема $i^+$ и множества негативных айтемов $\mathcal{I}^-$ функция потерь имеет вид:

$$
\mathcal{L}_{\text{BCE}} =
- \Big[
\log \sigma\bigl(r(u, i^+)\bigr)
+ \sum_{i^- \in \mathcal{I}^-}
\log \bigl(1 - \sigma(r(u, i^-))\bigr)
\Big]
$$

In [ ]:
class BCEModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_bce = BCEModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bce = torch.optim.Adam(params=model_bce.parameters(), lr=LEARNING_RATE)
best_checkpoint_bce = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bce,
    optimizer=optimizer_bce,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_bce.load_state_dict(best_checkpoint_bce)
final_metrics_bce = evaluation(
    yambda_eval_dataloader,
    model_bce,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bce_recs(final_metrics_bce)

## 6. BPR loss (1 балл)

Помимо BCE, двухбашенные модели также могут обучаться с использованием Bayesian Personalized Ranking (BPR).

Модель обучается на парах айтемов:

- позитивный айтем $i^+$, с которым пользователь действительно взаимодействовал;
- негативный айтем $i^-$, с которым пользователь не взаимодействовал (в данном подходе выбирается равномерно из каталога).

Цель обучения — добиться, чтобы для каждого пользователя выполнялось:

$$
r(u, i^+) > r(u, i^-)
$$

Таким образом, BPR напрямую приближает оптимизацию метрик ранжирования (Recall@K, nDCG@K), что делает его подходящим для retrieval-моделей.

#### Формула

Для каждого пользователя $u$ выбирается один позитивный айтем $i^+$ и один негативный айтем $i^-$. Функция потерь BPR определяется как:

$$
\mathcal{L}_{\text{BPR}}
= - \sum_{u \in \mathbf{U}} \log \sigma \bigl(r(u, i^+) - r(u, i^-)\bigr),
$$

где:
- $\mathbf{U}$ - набор польователей в батче;
- $r(u, i)$ — скор релевантности пользователя $u$ и айтема $i$;
- $\sigma(x) = \frac{1}{1 + e^{-x}}$ — сигмоида.

In [ ]:
class BPRModel(TwoTower):
  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_bpr = BPRModel(num_items=catalog_size, embedding_dim=64).to(DEVICE)
optimizer_bpr = torch.optim.Adam(params=model_bpr.parameters(), lr=LEARNING_RATE)
best_checkpoint_bpr = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_bpr,
    optimizer=optimizer_bpr,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_bpr.load_state_dict(best_checkpoint_bpr)
final_metrics_bpr = evaluation(
    yambda_eval_dataloader,
    model_bpr,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_bpr_recs(final_metrics_bpr)

## 7. Sampled softmax, uniform negatives (1 балл)

Sampled softmax — это аппроксимация полного softmax: вместо всех айтемов мы берём небольшой их набор и считаем softmax только по нему.

Для каждого пользователя $u$ у нас есть:
- позитивный айтем $i^+$;
- множество семплированных негативов $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$.

#### Формула

$$
\mathcal{L}_{\text{sampled-uniform}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

In [ ]:
class SampledUniformModel(TwoTower):
  def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_uniform = SampledUniformModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_uniform = torch.optim.Adam(params=model_sampled_uniform.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_uniform = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_uniform,
    optimizer=optimizer_sampled_uniform,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_sampled_uniform.load_state_dict(best_checkpoint_sampled_uniform)
final_metrics_sampled_uniform = evaluation(
    yambda_eval_dataloader,
    model_sampled_uniform,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_uniform_recs(final_metrics_sampled_uniform)

## 8. Sampled softmax, in-batch negatives (1 балл)

В прошлой задаче мы приближали полный softmax, сэмплируя негативы равновероятно из каталога. Теперь рассмотрим ещё более популярный подход: использование in-batch негативов.

Для каждого пользователя $u$ у нас есть:
- позитивный айтем $i^+$;
- множество семплированных негативов $\mathcal{N}(u) = \{i_1^-, \dots, i_K^-\}$ (только теперь мы семплируем не из всего каталога, а из батча).

#### Формула

$$
\mathcal{L}_{\text{sampled-batch}}(u) = - \log \frac{\exp(r(u, i^+))}{\exp(r(u, i^+)) + \sum_{i^- \in \mathcal{N}(u)}\exp(r(u, i^-))}.
$$

In [ ]:
class SampledInBatchModel(TwoTower):
  def __init__(self, num_items: int, embedding_dim: int, num_negatives: int) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_in_batch = SampledInBatchModel(num_items=catalog_size, embedding_dim=64, num_negatives=2048).to(DEVICE)
optimizer_sampled_in_batch = torch.optim.Adam(params=model_sampled_in_batch.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch,
    optimizer=optimizer_sampled_in_batch,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_sampled_in_batch.load_state_dict(best_checkpoint_sampled_in_batch)
final_metrics_sampled_in_batch = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_recs(final_metrics_sampled_in_batch)

## 9. Sampled softmax, in-batch negatives + logq correction (1 балл)

Использование in-batch подход это быстро и эффективно, но остаётся важная проблема: такое распределение негативов не совпадает с распределением в случае полного или uniform sampled softmax.

Один из стандартных способов избавиться от смещения — добавить log-q коррекцию.

#### Почему нужна коррекция

Обычный in-batch подход воспринимает все негативы как “равноправные”, но в реальности некоторые айтемы встречаются гораздо чаще, другие — почти никогда.

То есть негативы получаются как выборка из некоторого распределения $q(i)$, а не равномерные. Если мы хотим приблизиться к полному softmax, нужно компенсировать это смещение.

#### Формула

Пусть $q(i)$ — вероятность того, что айтем $i$ будет появляться как негатив-кандидат.

Тогда корректируем логит негативов:

$$
\tilde{r}(u,i) = r(u,i) - \log q(i),
$$

где $q(i)$ — вероятность появления айтема $i$ в качетстве негатива: частота его появления среди всех позитивов: $\frac{\#i}{\#all}$.

In [ ]:
def build_q_from_train_interactions(
  train_data: pl.DataFrame,
  catalog_size: int,
  item_col: str = "item_id",
  eps: float = 1e-12,
) -> torch.Tensor:
  #####################
  ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
  #####################
  pass

In [ ]:
class SampledInBatchModelLogQ(TwoTower):
  def __init__(
    self,
    num_items: int,
    embedding_dim: int,
    num_negatives: int,
    q: torch.Tensor,
    eps: float = 1e-12,
  ) -> None:
    super().__init__(num_items=num_items, embedding_dim=embedding_dim)
    self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
    self.num_negatives = num_negatives
    self.eps = eps

    q = q.detach().float()
    q = q / (q.sum() + eps)
    logq = torch.log(q.clamp_min(eps))
    self.register_buffer("logq", logq)

    self.init_weights(0.02)

  def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
    #####################
    ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
    #####################
    pass

In [ ]:
gc.collect()
torch.cuda.empty_cache()

#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################
q = build_q_from_train_interactions(...)
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

model_sampled_in_batch_logq = SampledInBatchModelLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq = torch.optim.Adam(params=model_sampled_in_batch_logq.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq,
    optimizer=optimizer_sampled_in_batch_logq,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_sampled_in_batch_logq.load_state_dict(best_checkpoint_sampled_in_batch_logq)
final_metrics_sampled_in_batch_logq = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_recs(final_metrics_sampled_in_batch_logq)

# Лидерборд и выводы

Собираем таблицу со всеми методами и метриками.

In [ ]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled softmax, uniform negatives", **final_metrics_sampled_uniform},
    {"method": "Sampled softmax, in-batch negatives", **final_metrics_sampled_in_batch},
    {"method": "Sampled softmax, in-batch negatives + logq correction", **final_metrics_sampled_in_batch_logq},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard

## 10. Вопросы на понимание (1 балл)

1. В чем основная проблема использования `Full softmax`?
2. Почему `BCE` хуже показал себя чем `BPR`?
3. В чем может быть проблема с `Sampled softmax, uniform` подходом?
4. В чем проблема `in-batch` подхода без использования `logq`-коррекции?
5. Почему при добавлении `logq` у нас упал `coverage`?

Ответы - текстом

# Бонусные задания

Вы уже реализовали основные подходы, провели замеры и сделали выводы о том, как разные функции потерь и стратегии негативного сэмплирования влияют на качество модели. В качестве бонусных заданий предлагается реализовать более продвинутые методы, об одном которых мы также говорили на лекции.

## 11. Sampled softmax, in-batch negatives + **fixed** logq correction (1 балл)

В этом бонусном задании реализуем более аккуратный вариант `logq correction`, предложенный в статье *Correcting the LogQ Correction: Revisiting Sampled Softmax for Large-Scale Retrieval*. Стандартная `logq`-коррекция не полностью устраняет смещение, возникающее из-за неравномерного появления объектов в батче.

Ключевая идея состоит в том, что в стандартном выводе `logq` положительный объект неявно трактуется так, будто он был получен из того же распределения, что и негативы. На практике это не так: положительный объект всегда присутствует в примере детерминированно и не является случайно выбранным негативом. Именно эта деталь приводит к дополнительному смещению.

Реализуйте **fixed logq correction**: исправленный вариант коррекции, который учитывает, что политивный пример не должен обрабатываться так же, как семплированные негативы.

In [ ]:
class SampledInBatchModelFixedLogQ(TwoTower):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q: torch.Tensor,
        eps: float = 1e-12,
    ) -> None:
        super().__init__(num_items=num_items, embedding_dim=embedding_dim)
        self.encoder = UserEncoder(num_items=num_items, embedding_dim=embedding_dim)
        self.num_negatives = num_negatives
        self.eps = eps

        q = q.detach().float()
        q = q / q.sum()
        self.register_buffer("q", q)

        self.init_weights(0.02)

    def compute_loss(self, user_repr: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        #####################
        ### (づ•̀ᴗ•́)づ──☆*:・ﾟ
        #####################
        pass

In [ ]:
gc.collect()
torch.cuda.empty_cache()

model_sampled_in_batch_logq_fixed = SampledInBatchModelFixedLogQ(num_items=catalog_size, embedding_dim=64, num_negatives=2048, q=q).to(DEVICE)
optimizer_sampled_in_batch_logq_fixed = torch.optim.Adam(params=model_sampled_in_batch_logq_fixed.parameters(), lr=LEARNING_RATE)
best_checkpoint_sampled_in_batch_logq_fixed = train(
    train_dataloader=yambda_train_dataloader,
    valid_dataloader=yambda_eval_dataloader,
    model=model_sampled_in_batch_logq_fixed,
    optimizer=optimizer_sampled_in_batch_logq_fixed,
    num_epochs=NUM_EPOCHS,
    catalog_size=catalog_size,
    topk=TOPK,
    device=DEVICE
)

In [ ]:
model_sampled_in_batch_logq_fixed.load_state_dict(best_checkpoint_sampled_in_batch_logq_fixed)
final_metrics_sampled_in_batch_logq_fixed = evaluation(
    yambda_eval_dataloader,
    model_sampled_in_batch_logq_fixed,
    catalog_size=catalog_size,
    topk=TOPK
)
tests.check_softmax_inbatch_logq_fixed_recs(final_metrics_sampled_in_batch_logq_fixed)

## 12. Улучшение аггрегации история пользователя (1 балл)

В этом бонусном задании вам предлагается самостоятельно улучшить способ агрегации истории пользователя в модели и добиться дополнительного прироста качества. В базовых решениях история пользователя уже используется для построения пользовательского представления, однако сама схема агрегации может быть довольно простой и не всегда позволяет достаточно хорошо учитывать порядок, важность и контекст прошлых взаимодействий.

Цель этого задания — получить **дополнительный прирост качества не менее чем на 0.01 по nDCG в абсолютных значениях** по сравнению с вашим лучшим решением из предыдущих пунктов. Иными словами, если ваш лучший результат раньше был, например, `nDCG@K = 0.123`, то для выполнения этого бонусного задания нужно получить как минимум `0.133`.

Важно: этот пункт проверяющие **не проверяли заранее самостоятельно**, поэтому дополнительны балл будет выставляться только за тот код, который действительно является **воспроизводимым** в **Google Colab на GPU T4** и получить такие же или очень близкие результаты. Поэтому в решении особенно важно:
- зафиксировать сиды
- явно указать все изменения в модели
- сохранить корректный и полный пайплайн обучения
- не опускать важные ячейки с подготовкой данных, обучением и оценкой


In [ ]:
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################
final_metrics_your_solution = ...
#####################
### (づ•̀ᴗ•́)づ──☆*:・ﾟ
#####################

## Лидерборд с бонусами

In [ ]:
leaderboard = pl.DataFrame([
    {"method": "Softmax loss", **final_metrics_full},
    {"method": "BCE loss", **final_metrics_bce},
    {"method": "BPR loss", **final_metrics_bpr},
    {"method": "Sampled, uniform", **final_metrics_sampled_uniform},
    {"method": "Sampled, in-batch", **final_metrics_sampled_in_batch},
    {"method": "Sampled, in-batch + logq", **final_metrics_sampled_in_batch_logq},
    {"method": "Sampled, in-batch + fixed logq", **final_metrics_sampled_in_batch_logq_fixed},
    {"method": "Your custom solution", **final_metrics_your_solution},
])

leaderboard = leaderboard.sort(["recall", "ndcg"], descending=True)
leaderboard